In [4]:
import pandas as pd
import numpy as np
import networkx as nx
import yfinance as yf

# Download selected tickers directly from Yahoo (real data)
tickers = ["NVDA", "AAPL", "MSFT", "TSLA"]
prices = yf.download(
    tickers,
    start="2020-01-01",
    end=None,
    interval="1d",
    auto_adjust=False,
    progress=False,
    group_by="column",
)

adj = prices["Adj Close"]
adj.index = pd.to_datetime(adj.index)
adj.sort_index(inplace=True)
print(adj.shape)
print(adj.index.min(), adj.index.max())

# Compute log returns
rets = np.log(adj / adj.shift(1)).dropna(how="all")

# Correlation -> distance -> MST -> degree centrality ranking
corr = rets.corr()
dist = np.sqrt(2 * (1 - corr))
np.fill_diagonal(dist.values, 0.0)

G = nx.Graph()
names = dist.index.tolist()
for i, ti in enumerate(names):
    for j in range(i + 1, len(names)):
        w = dist.iat[i, j]
        if not np.isnan(w):
            G.add_edge(ti, names[j], weight=w)

mst = nx.minimum_spanning_tree(G, weight="weight")
deg = nx.degree_centrality(mst)
ranking = pd.Series(deg, name="degree_centrality").sort_values()
print("MST degree centrality ranking (low = periphery):")
print(ranking)



4 Failed downloads:
['AAPL', 'NVDA', 'TSLA', 'MSFT']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')


(0, 4)
NaT NaT
MST degree centrality ranking (low = periphery):
Series([], Name: degree_centrality, dtype: object)
